# 🎙️ TTS Text Optimizer for DesiVocal.com

**Optimize translated text with proper punctuation for natural voice generation.**

This notebook allows you to:
1.  **Setup Ollama**: Install and run Ollama locally in Colab.
2.  **Download Model**: Choose and pull a high-quality LLM (e.g., Qwen2.5, TranslateGemma).
3.  **Optimize Text**: Input your text, and the AI will add proper punctuation for natural TTS flow.
4.  **Download Result**: Save the optimized text as a `.txt` file for use on DesiVocal.com.

## 📦 Step 1: Install & Setup Ollama
Run this cell to install Ollama and start the server in the background.

In [ ]:
# Install required packages
!pip install -q ollama requests ipywidgets

# Install and start Ollama server
import subprocess
import time
import os
import sys

print("🦙 Installing Ollama...")

# Install zstd first (required for Ollama extraction)
!apt-get update -qq && apt-get install -y -qq zstd > /dev/null 2>&1

# Download and install Ollama
!curl -fsSL https://ollama.com/install.sh | sh

print("\n🚀 Starting Ollama server in background...")

# Start Ollama server in background
os.environ['OLLAMA_HOST'] = '127.0.0.1:11434'
subprocess.Popen(['/usr/local/bin/ollama', 'serve'], stdout=subprocess.DEVNULL, stderr=subprocess.DEVNULL)

# Wait for server to start
time.sleep(5)

# Verify server is running
try:
    import ollama
    ollama.list()
    print("✅ Ollama server is running and ready!")
except Exception as e:
    print(f"⚠️ Ollama server may not be ready yet. Error: {e}")
    print("   Please wait a few seconds and try running the next cell.")

## 📥 Step 2: Download Model
Select the model you want to use for optimization. `qwen2.5:14b` is recommended for high quality.

In [ ]:
import ipywidgets as widgets
from IPython.display import display
import ollama

print("🦙 Ollama Model Selection")
print("=" * 30)

# Model options
OLLAMA_MODELS = {
    "qwen2.5:14b (Recommended - High Quality)": "qwen2.5:14b",
    "qwen2.5:7b (Faster)": "qwen2.5:7b",
    "translategemma:27b (Very High Quality, Large)": "translategemma:27b",
    "mistral:7b (Good Quality)": "mistral:7b",
    "gemma2:9b (Google's Best Format)": "gemma2:9b"
}

model_dropdown = widgets.Dropdown(
    options=list(OLLAMA_MODELS.keys()),
    value="qwen2.5:14b (Recommended - High Quality)",
    description='Model:',
    style={'description_width': 'initial'},
    layout=widgets.Layout(width='400px')
)

display(model_dropdown)
print("\nSelect a model and run this cell to download it.")

In [ ]:
# Pull the selected model
selected_model_name = OLLAMA_MODELS[model_dropdown.value]
print(f"📥 Pulling model: {selected_model_name}...")
print("   This may take a few minutes.")

try:
    # Pull with stream to show progress (simplified for non-interactive output)
    current_digest = ''
    for progress in ollama.pull(selected_model_name, stream=True):
        digest = progress.get('digest', '')
        if digest != current_digest and current_digest:
             print() # Newline
        current_digest = digest
        
        status = progress.get('status', '')
        if 'completed' in progress and 'total' in progress:
             completed = progress['completed']
             total = progress['total']
             pct = (completed / total * 100) if total > 0 else 0
             print(f"\r   {status}: {pct:.1f}%", end='', flush=True)
        else:
             print(f"\r   {status}", end='', flush=True)

    print(f"\n\n✅ Model '{selected_model_name}' ready to use!")
except Exception as e:
    print(f"\n❌ Error pulling model: {e}")

## 🧠 Step 3: Define Optimizer Class
This code defines the logic to communicate with Ollama and optimize the text.

In [ ]:
import requests
import json
import sys

class TTSOptimizer:
    """Optimizes text for desivocal.com TTS generation"""
    
    def __init__(self, model_name="qwen2.5:14b"):
        self.ollama_url = "http://localhost:11434/api/generate"
        self.model = model_name
        print(f"🤖 Initialized TTS Optimizer with model: {self.model}")

    def get_optimization_prompt(self, text: str, language: str = "Hindi") -> str:
        prompt = f"""You are a TTS punctuation expert optimizing text for natural voice generation on desivocal.com.

━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
CRITICAL TASK
━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
Add PROPER PUNCTUATION to make this {language} text sound NATURAL when read by TTS voice-over system.

DESIVOCAL.COM BEST PRACTICES (MANDATORY):
━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
1. ✓ AMPLE PUNCTUATION for natural pauses and modulation:
   - Use periods (.) to break sentences (max 15-20 words per sentence)
   - Use commas (,) for natural breathing pauses (every 8-12 words)
   - Use question marks (?) for questions
   - Use exclamation marks (!) for emphasis/excitement

2. ✓ MULTIPLE PUNCTUATIONS for emotion/expression:
   - ??? for strong doubt/confusion/repeated questions
   - !!! for excitement/shock/strong emotion
   - ... for hesitation/suspense/trailing off

3. ✓ ABBREVIATIONS with dots:
   - AI → A.I., PhD → Ph.D., etc.

4. ✓ SENTENCE BREAKING (CRITICAL):
   - Break long sentences into shorter ones (10-20 words max)

━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
STRICT RULES (DO NOT VIOLATE):
━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
✗ DO NOT change ANY words
✗ DO NOT add or remove content
✗ DO NOT translate anything
✗ DO NOT use SSML tags

✓ ONLY add punctuation marks: . , ? ! ??? !!! ...
✓ Keep 100% of original words intact

━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
INPUT TEXT:
━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
{text}

━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
OUTPUT (TTS-OPTIMIZED VERSION):
━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
Return ONLY the optimized text with proper punctuation. No explanations."""
        return prompt
    
    def optimize(self, text: str, language: str ="Hindi") -> str:
        prompt = self.get_optimization_prompt(text, language)
        
        payload = {
            "model": self.model,
            "prompt": prompt,
            "stream": False,
            "options": {
                "temperature": 0.4,
                "top_p": 0.9,
                "num_predict": -1
            }
        }
        
        print(f"📤 Sending to Ollama ({self.model})...")
        
        try:
            response = requests.post(self.ollama_url, json=payload, timeout=300)
            response.raise_for_status()
            result = response.json()
            optimized_text = result.get("response", "").strip()
            # Clean output
            optimized_text = self._clean_output(optimized_text)
            print("✅ Optimization complete!")
            return optimized_text
        except Exception as e:
            print(f"❌ Error: {e}")
            return None

    def _clean_output(self, text: str) -> str:
        text = text.replace("```", "").replace("**", "")
        lines = [line.strip() for line in text.split('\n') 
                 if line.strip() and not line.strip().startswith('#') and not line.strip().startswith('OUTPUT')]
        return '\n'.join(lines).strip()

## 📝 Step 4: Run Optimization
Enter your text below and run the cell to optimize it.

In [ ]:
from google.colab import files

# Text Input Widget
input_text_widget = widgets.Textarea(
    value='',
    placeholder='Paste your text here...',
    description='Text:',
    layout=widgets.Layout(width='100%', height='200px')
)

language_input = widgets.Text(
    value='Hindi',
    placeholder='Target Language',
    description='Language:',
    layout=widgets.Layout(width='300px')
)

display(language_input)
display(input_text_widget)

In [ ]:
# Run Optimization
if not input_text_widget.value.strip():
    print("⚠️ Please enter some text above first!")
else:
    # Initialize optimizer with selected model
    # using the variable from Step 2
    try:
        model_to_use = selected_model_name
    except NameError:
        model_to_use = "qwen2.5:14b" # Fallback

    optimizer = TTSOptimizer(model_name=model_to_use)
    
    print("\n⏳ Optimizing text...")
    optimized_text = optimizer.optimize(input_text_widget.value, language=language_input.value)
    
    if optimized_text:
        print("\n✨ Optimized Text Result:")
        print("=" * 40)
        print(optimized_text)
        print("=" * 40)
        
        # Save to file
        output_filename = 'tts_optimized_output.txt'
        with open(output_filename, 'w', encoding='utf-8') as f:
            f.write(optimized_text)
            
        print(f"\n💾 Saved to {output_filename}")
        
        # Trigger download
        files.download(output_filename)